### K?t?phaneleri ??eri Aktarma

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
from DataMacpy import *

### Veriyi ?n ??leme

In [ ]:
# Veriyi i?eri aktaral?m.

# veri yolumuzu verelim
file_path = 'pokemon.csv'

# verimizi okuyoruz. 
pokemon_data = pd.read_csv(file_path)

print("Setup Complete")

In [ ]:
import seaborn as sns
sns.heatmap(pokemon_data.isnull(),yticklabels=False,cbar=False,cmap='viridis')

In [ ]:
# Veri G?rselle?tirme

pokemon_data.tail(5)

In [ ]:
# hedef kolonumuzu belirleyelim.
y_not_enc = pokemon_data.Type_1
y_not_enc

In [ ]:
# target verilerimiz string oldu?u i?in LabelEncoder yap?yoruz. Her s?n?f? bir say?ya at?yoruz.
l_fit = LabelEncoder().fit(y_not_enc)
y = l_fit.transform(y_not_enc)
#y = pd.DataFrame(y, columns = ["Type_1"]) # dataframe'e ?evirmek i?in bu sat?r? kullanabiliriz.
y[0:20]

In [ ]:
# labellerimizi buluyoruz.
y_label = l_fit.inverse_transform([i for i in range(max(y)+1)])
label_and_num = list(zip(y_label, [i for i in range(max(y)+1)]))
label_and_num

In [ ]:
# ?zniteliklerimizi belirleyelim.
X = pokemon_data
X.drop(["Number", "Type_1"], axis=1, inplace=True) # t?m verilerimizden hedef kolonumuzu ??kar?yoruz.
# Number kolonumuzu da gereksiz oldu?u e?itimde gereksiz oldu?u i?in ??kar?yoruz.
X2 = X.copy() # Burada bir X2 e?itim seti daha ??kar?p onda da farkl? veri ?n i?lemeleri yaparak se?ti?imiz
# model ?zerinden e?itim yaparak hangi veri ?n i?leme iyi olmu? ona bakarak se?im yapaca??z
X

In [ ]:
# DataMacpy k?t?phanemizdeki fonksiyon ile eksik veriye sahip kolonlar? buluyoruz.
cols_with_missing = find_missing_cols(X)
cols_with_missing

In [ ]:
# Eksik bilgileri d???r?yoruz.
print(f"Eksik veriler d???r?lmeden ?nce verilerimizin boyutu: {X.shape}")
X.drop(cols_with_missing, axis=1, inplace=True)
print(f"Eksik veriler d???r?ld?kten sonra verilerimizin boyutu: {X.shape}")

X2.drop(['Type_2','Egg_Group_2'], axis=1, inplace=True)
print(f"Eksik veriler d???r?ld?kten sonra ?nce X2 verilerimizin boyutu: {X2.shape}")

In [ ]:
# DataMacpy k?t?phanemizdeki fonksiyon ile benzersiz birimi 15'den fazla olan veri tipi object i?eren kolonlar? buluyoruz.
object_cols, d = calc_object_cols(X, 15)
print(f"object_cols: {object_cols}\n\n d: {d}")

# Bu kolonlar? d???r?yoruz ??nk? her veriye ?zel bir benzersiz bilgi i?eriyorsa overfitting olabilir.
# E?er d?n??t?rmeye kalkarsak her benzersiz birim i?in OneHotEncoder yaparsak ?ok fazla kolona sahip veri seti elde ederiz.
X = X.drop(object_cols, axis = 1)
X2 = X2.drop(object_cols, axis = 1)

In [ ]:
# DataMacpy k?t?phanemizdeki fonksiyon ile benzersiz birimi 0'dan fazla olan veri tipi object i?eren
# kolonlar? buluyoruz.
object_cols, d = calc_object_cols(X, 0)
print(f"object_cols X: {object_cols}\n\n d: {d}")

object_cols2, d2 = calc_object_cols(X2, 0)
print(f"\n\nobject_cols X2: {object_cols2}\n\n d: {d2}")

In [ ]:
# Verilerimize OrdinalEncoder uyguluyoruz. OneHot denemesi yap?lacak!
ordinal_encoder = OrdinalEncoder()
X[object_cols] = ordinal_encoder.fit_transform(X[object_cols])
X

In [ ]:
sns.heatmap(X2.isnull(),yticklabels=False,cbar=False,cmap='viridis')

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesClassifier


# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='constant')


# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, ["Pr_Male"]),
        ('cat', categorical_transformer, object_cols2)
    ])

# Define model
model = ExtraTreesClassifier(bootstrap=True, ccp_alpha=0.0,
                          criterion='gini', max_depth=650,
                          max_features=None, max_leaf_nodes=1000,
                          min_impurity_decrease=0.0,
                          min_samples_leaf=1, min_samples_split=3,
                          min_weight_fraction_leaf=0.0,
                          n_estimators=100)

# Bundle preprocessing and modeling code in a pipeline
clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', model)
                     ])

In [ ]:
# Verilerimizi train ve validation olarak ay?r?yoruz.
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)
X_train2, X_valid2, y_train2, y_valid2 = train_test_split(X2, y, train_size=0.8, test_size=0.2, random_state=0)

### Default de?erler i?in Model se?imi yap?yoruz

In [ ]:
model_fit_and_predict(ExtraTreesClassifier(bootstrap=True, ccp_alpha=0.0,
                          criterion='gini', max_depth=650,
                          max_features=None, max_leaf_nodes=1000,
                          min_impurity_decrease=0.0,
                          min_samples_leaf=1, min_samples_split=3,
                          min_weight_fraction_leaf=0.0,
                          n_estimators=100), y_label, X_train, y_train, X_valid, y_valid)

In [ ]:
model_fit_and_predict(clf, y_label, X_train2, y_train2, X_valid2, y_valid2)

In [ ]:
preprocessor

In [ ]:
categorical_transformer